# SignalRank — Retrieval Evaluation

Compares **embedding-only vs BM25 vs hybrid (RRF) vs hybrid+cross-encoder** on 500 jobs with graded qrels (0/1/2).
Metrics: **P@K, R@K, MRR, nDCG@K**. Runs offline without Docker/DB via TF-IDF fallback.

Kaggle: `ahmedarfaoui/signalrank-jobs-500` · HF: `ahmedarfaoui/signalrank-jobs`

In [ ]:
import json, pathlib, sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Ensure backend is importable
sys.path.insert(0, str(Path('backend').resolve()))
sys.path.insert(0, str(Path('.').resolve()))

jobs_path = 'data/raw/jobs.jsonl'
qrels_path = 'data/qrels.jsonl'
cv_path = 'data/sample/cv_sample.txt'

jobs = [json.loads(l) for l in open(jobs_path, encoding='utf-8') if l.strip()]
print(f"{len(jobs)} jobs")
print(open(cv_path, encoding='utf-8').read()[:400])

In [ ]:
!python backend/app/evaluation/compare.py --jobs data/raw/jobs.jsonl --qrels data/qrels.jsonl --cv data/sample/cv_sample.txt --mode hybrid-only --out artifacts/metrics.json
!cat artifacts/metrics.json

In [ ]:
import json
m = json.loads(open('artifacts/metrics.json', encoding='utf-8').read())
df = pd.DataFrame(m['methods']).T
df[['precision@10','recall@10','mrr','ndcg@10','ndcg@5']].round(3)


In [ ]:
df[['ndcg@10','ndcg@5']].plot(kind='bar', figsize=(8,4), title='nDCG by method (higher is better)')
plt.xticks(rotation=0); plt.tight_layout(); plt.savefig('artifacts/ndcg.png', dpi=150); plt.show()

### With cross-encoder (needs `sentence-transformers`)
If available, reranks top-100 with `cross-encoder/ms-marco-MiniLM-L-6-v2`.

In [ ]:
# Full eval (may download ~80MB model)
!python backend/app/evaluation/compare.py --jobs data/raw/jobs.jsonl --qrels data/qrels.jsonl --cv data/sample/cv_sample.txt --with-ce --out artifacts/metrics-full.json
!cat artifacts/metrics-full.json